# Ejercicio 03-01-basic-anthropic

## Objetivo

Utilizar Google Collab para crear un código Python que conecte con el API del LLM de Anthropic y permita obtener una respuesta

## Requisitos previos

* Cuenta activa en Anthropic Console, con créditos y con un API Key generada
* Cuenta de Google Personal o Corporativa
* Configuración de los secretos de la API Keys de Anthropic en Google Collab
  * Se habría realizado en el módulo 00-getting-started, pero si no se ha realizado, se pueden configurar en este ejercicio siguiendo esas instrucciones

# Configuración

## Paso 1: Instalar Dependencias

Pasos a seguir:

* Instalar la SDK de Anthropic
  * https://pypi.org/project/anthropic/
* Verificar la versión de instalación de la SDK


In [1]:
# Install SDK "Anthropic"
!pip install anthropic

# Verify installation
import anthropic

print(f"✅ anthropic {anthropic.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.5/837.5 kB 13.7 MB/s eta 0:00:00
✅ anthropic 0.105.2


## Paso 2: Cargar la API Key

Pasos a seguir:

* Acceder a los secretos desde Google Collab
* Identificar el API Key de Acceso de Anthropic
* Mapear el secreto de Google Collab con la variable de entorno de OpenAI



In [2]:
# Prepare imports
import os
from google.colab import userdata

# Load API keys from Colab Secrets for Anthropic
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY_LAB")

print("✅ Anthropic API Key loaded successfully")

✅ Anthropic API Key loaded successfully


## Paso 3: Crear un Cliente de Anthropic

Pasos a seguir:

* Utilizar el constructor de la SDK de Anthropic con los parámetros necesarios

Nota: No se añadirá ningún parámetro extra para este caso

In [3]:
# Prepare imports
from anthropic import Anthropic

# Create client for Anthropic
anthropic_client = Anthropic()

print("✅ Client Anthropic configured successfully")

✅ Client Anthropic configured successfully


# Helpers

Se han definido una serie de funciones de soporte para el desarrollo de la parte técnica y los ejemplos utilizados

* Funciones de soporte a metadatos
* Funciones de soporte a costes

## Paso 1: Cargar funciones de ayuda para obtener metainformación de la respuesta en Anthropic

Se puede obtener diferente información sobre lo que ha ocurrido en la respuesta a un LLM: modelo utilizado, tokens de entrada / salida, etc.

**Importante:** Suele depender del tipo de respuesta que facilite el API del proveedor con el que se ha consultado

In [4]:
import json

# ************************
#  Helpers
# ************************

def safe_get(obj, attr, default=None):
    if obj is None:
        return default

    if isinstance(obj, dict):
        return obj.get(attr, default)

    return getattr(obj, attr, default)


def get_anthropic_text_response(response) -> str:
    """
    Extracts text from an Anthropic Message response.
    Anthropic returns content as a list of blocks.
    """
    content_blocks = safe_get(response, "content", []) or []

    text_parts = []

    for block in content_blocks:
        block_type = safe_get(block, "type")
        block_text = safe_get(block, "text")

        if block_type == "text" and block_text:
            text_parts.append(block_text)

    return "\n".join(text_parts)


# ************************
#  Basic - Anthropic
# ************************

def get_response_metadata_basic_anthropic(response) -> dict:
    usage = safe_get(response, "usage")

    input_tokens = safe_get(usage, "input_tokens", 0) or 0
    output_tokens = safe_get(usage, "output_tokens", 0) or 0

    total_tokens = input_tokens + output_tokens

    return {
        "metadata": {
            "provider": "anthropic",
            "id": safe_get(response, "id"),
            "model": safe_get(response, "model"),
            "usage": {
                "input_tokens": input_tokens,
                "output_tokens": output_tokens,
                "total_tokens": total_tokens
            }
        }
    }


# ************************
#  Generic - Anthropic
# ************************

def get_response_metadata_anthropic(response) -> dict:
    usage = safe_get(response, "usage")

    input_tokens = safe_get(usage, "input_tokens", 0) or 0
    output_tokens = safe_get(usage, "output_tokens", 0) or 0

    cache_creation_input_tokens = safe_get(
        usage,
        "cache_creation_input_tokens",
        0
    ) or 0

    cache_read_input_tokens = safe_get(
        usage,
        "cache_read_input_tokens",
        0
    ) or 0

    total_input_tokens = (
        input_tokens +
        cache_creation_input_tokens +
        cache_read_input_tokens
    )

    total_tokens = total_input_tokens + output_tokens

    service_tier = safe_get(usage, "service_tier")
    inference_geo = safe_get(usage, "inference_geo")
    server_tool_use = safe_get(usage, "server_tool_use")

    return {
        "metadata": {
            "provider": "anthropic",
            "id": safe_get(response, "id"),
            "model": safe_get(response, "model"),
            "type": safe_get(response, "type"),
            "role": safe_get(response, "role"),

            # Anthropic does not expose status like OpenAI Responses API
            "status": None,

            # Anthropic Message response does not normally expose created_at
            "created_at": None,

            # Anthropic-specific finish metadata
            "stop_reason": safe_get(response, "stop_reason"),
            "stop_sequence": safe_get(response, "stop_sequence"),

            "usage": {
                # Normalized total input tokens
                "input_tokens": total_input_tokens,

                # Anthropic-specific input token breakdown
                "input_non_cached_tokens": input_tokens,
                "input_cache_creation_tokens": cache_creation_input_tokens,
                "input_cached_tokens": cache_read_input_tokens,

                # Output tokens
                "output_tokens": output_tokens,

                # Anthropic does not expose output reasoning tokens
                # in the same way as OpenAI output_tokens_details.
                "output_visible_tokens": output_tokens,
                "output_reasoning_tokens": 0,

                # Calculated total
                "total_tokens": total_tokens,

                # Optional Anthropic metadata
                "service_tier": service_tier,
                "inference_geo": inference_geo,
                "server_tool_use": server_tool_use
            }
        }
    }


print("✅ Anthropic response metadata functions loaded")

✅ Anthropic response metadata functions loaded


## Paso 2: Cargar funciones para obtener los costes de la respuesta en Anthropic

Se aconseja acceder a las tarifas del proveedor

**Importante**: Suelen cambiar sin avisar

Recursos:

* Anthropic: https://platform.claude.com/docs/en/about-claude/pricing


In [5]:
PRICING = {
    # Anthropic (USD per 1M tokens)
    "claude-haiku-4-5-20251001": {"input": 1.00,"output": 5.00},
    "claude-sonnet-4-6": {"input": 3.00,"output": 15.00},
    "claude-opus-4-7": {"input": 5.00,"output": 25.00},
}

def get_model_prices(model):
    prices = PRICING.get(model)

    if prices:
        return prices

    for key in PRICING:
        if key in model or model in key:
            return PRICING[key]

    return None

def money(value, decimals=2):
    return f"${value:,.{decimals}f}"

def safe_get(obj, attr, default=None):
    if obj is None:
        return default

    if isinstance(obj, dict):
        return obj.get(attr, default)

    return getattr(obj, attr, default)

# ************************
#  Basic
# ************************

def get_calculate_cost_basic_anthropic(model, input_tokens, output_tokens):
    prices = get_model_prices(model)

    if not prices:
        return None

    input_tokens_cost_usd = (input_tokens / 1_000_000) * prices["input"]
    output_tokens_cost_usd = (output_tokens / 1_000_000) * prices["output"]

    total_tokens_cost_usd = input_tokens_cost_usd + output_tokens_cost_usd

    return {
        "cost": {
            "currency_type": "USD",
            "model": model,
            "input": round(input_tokens_cost_usd, 7),
            "output": round(output_tokens_cost_usd, 7),
            "total": round(total_tokens_cost_usd, 7)
        }
    }


# ************************
#  Advanced
# ************************

def get_calculate_cost_advance_anthropic(
    model,
    input_tokens,
    output_tokens,
    cache_creation_input_tokens=0,
    cache_read_input_tokens=0
):
    prices = get_model_prices(model)

    if not prices:
        return None

    input_tokens = input_tokens or 0
    output_tokens = output_tokens or 0
    cache_creation_input_tokens = cache_creation_input_tokens or 0
    cache_read_input_tokens = cache_read_input_tokens or 0

    # By default, cache tokens are priced as normal input tokens.
    # If you want specific cache pricing later, you can add:
    # "cache_creation_input": ...
    # "cache_read_input": ...
    input_price = prices["input"]
    output_price = prices["output"]

    cache_creation_price = prices.get("cache_creation_input", input_price)
    cache_read_price = prices.get("cache_read_input", input_price)

    input_tokens_cost_usd = (
        input_tokens / 1_000_000
    ) * input_price

    cache_creation_cost_usd = (
        cache_creation_input_tokens / 1_000_000
    ) * cache_creation_price

    cache_read_cost_usd = (
        cache_read_input_tokens / 1_000_000
    ) * cache_read_price

    output_tokens_cost_usd = (
        output_tokens / 1_000_000
    ) * output_price

    total_input_cost_usd = (
        input_tokens_cost_usd +
        cache_creation_cost_usd +
        cache_read_cost_usd
    )

    total_tokens_cost_usd = total_input_cost_usd + output_tokens_cost_usd

    total_input_tokens = (
        input_tokens +
        cache_creation_input_tokens +
        cache_read_input_tokens
    )

    total_tokens = total_input_tokens + output_tokens

    return {
        "cost": {
            "currency_type": "USD",
            "model": model,

            "input": round(total_input_cost_usd, 7),
            "input_non_cached": round(input_tokens_cost_usd, 7),
            "input_cache_creation": round(cache_creation_cost_usd, 7),
            "input_cached_read": round(cache_read_cost_usd, 7),

            "output": round(output_tokens_cost_usd, 7),
            "total": round(total_tokens_cost_usd, 7)
        }
    }


# ************************
#  Generic
# ************************

def get_calculate_cost_anthropic(response):
    model = safe_get(response, "model")
    usage = safe_get(response, "usage")

    prices = get_model_prices(model)

    if not prices:
        return None

    input_tokens = safe_get(usage, "input_tokens", 0) or 0
    output_tokens = safe_get(usage, "output_tokens", 0) or 0

    cache_creation_input_tokens = safe_get(
        usage,
        "cache_creation_input_tokens",
        0
    ) or 0

    cache_read_input_tokens = safe_get(
        usage,
        "cache_read_input_tokens",
        0
    ) or 0

    return get_calculate_cost_advance_anthropic(
        model=model,
        input_tokens=input_tokens,
        output_tokens=output_tokens,
        cache_creation_input_tokens=cache_creation_input_tokens,
        cache_read_input_tokens=cache_read_input_tokens
    )


print("✅ Anthropic cost calculation functions loaded")

✅ Anthropic cost calculation functions loaded


# Invocación de una llamada básica


## Paso 1: Llamada básica + Respuesta

Genera una respuesta general que NO esta sujeta a ningún tipo de parámetro

NO se utilizará ningún "system prompt"

**Importante**: Esto afecta al coste y a la calidad

Recursos de la respuesta:

* xxx

In [ ]:
# Define user prompt
USER_PROMPT = "What should I consider when modernizing an application?"

print("***** Llamada básica *****")

# Invoke LLM
response = anthropic_client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": USER_PROMPT}
    ]
)

# Get text response
response_text = response.content[0].text

# Show
print("Respuesta del modelo:\n")
print(response_text)

***** Llamada básica *****
Respuesta del modelo:

# Key Considerations for Modernizing an Application

## Architecture & Design
- **Current state assessment** – Document existing code, dependencies, and technical debt
- **Target architecture** – Decide on monolith, microservices, serverless, or hybrid approach
- **Migration strategy** – Plan big bang vs. incremental refactoring
- **API design** – Establish clean interfaces between components

## Technology Stack
- **Framework/language choices** – Balance innovation with team expertise
- **Database strategy** – Consider schema migration, data consistency, and scaling
- **Infrastructure** – Cloud provider, containerization, orchestration (Kubernetes, etc.)
- **Third-party services** – Evaluate build vs. buy for new capabilities

## Business & Risks
- **Timeline & budget** – Modernization is expensive; set realistic expectations
- **Business continuity** – Minimize disruption; maintain service during transition
- **Team capacity** – Do yo

## Paso 2: Obtener metadatos

### Modo 1: Mostrar metadatos básicos

In [ ]:
# Get metadata basic response
metadata_json = get_response_metadata_basic_anthropic(response)

# Show metadata
print("***** Basic Metadata *****")
print(json.dumps(metadata_json, indent=2, ensure_ascii=False))

***** Basic Metadata *****
{
  "metadata": {
    "provider": "anthropic",
    "id": "msg_01N4eu9fNMnXVKnu2SKzYxuX",
    "model": "claude-haiku-4-5-20251001",
    "usage": {
      "input_tokens": 17,
      "output_tokens": 379,
      "total_tokens": 396
    }
  }
}


### Modo 2: Mostrar metadatos

In [ ]:
# Get metadata advance response
metadata_json = get_response_metadata_anthropic(response)

# Show metadata
print("***** Metadata *****")
print(json.dumps(metadata_json, indent=2, ensure_ascii=False))

***** Metadata *****
{
  "metadata": {
    "provider": "anthropic",
    "id": "msg_01N4eu9fNMnXVKnu2SKzYxuX",
    "model": "claude-haiku-4-5-20251001",
    "type": "message",
    "role": "assistant",
    "status": null,
    "created_at": null,
    "stop_reason": "end_turn",
    "stop_sequence": null,
    "usage": {
      "input_tokens": 17,
      "input_non_cached_tokens": 17,
      "input_cache_creation_tokens": 0,
      "input_cached_tokens": 0,
      "output_tokens": 379,
      "output_visible_tokens": 379,
      "output_reasoning_tokens": 0,
      "total_tokens": 396,
      "service_tier": "standard",
      "inference_geo": "not_available",
      "server_tool_use": null
    }
  }
}


## Paso 3: Obtener costes

### Modo 1: Cálculo básico SIN función de costes específica

In [ ]:
# Pricing and Conditions
# - Estimated pricing for claude-haiku-4-5 (USD per 1M tokens)
# - Cached tokens are not included
# - Adjust these values ​​according to the current official pricing
INPUT_PRICE_PER_MILLION = 1.00
OUTPUT_PRICE_PER_MILLION = 5.00

# Cost calculation
input_cost = (response.usage.input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
output_cost = (response.usage.output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
total_cost = input_cost + output_cost

# Show
print("\n***** COST *****")
print(f"Input cost:  ${input_cost:.6f}")
print(f"Output cost: ${output_cost:.6f}")
print(f"Total cost:  ${total_cost:.6f}")


***** COST *****
Input cost:  $0.000017
Output cost: $0.001895
Total cost:  $0.001912


### Modo 2: Cálculo básico CON función de costes específica

In [ ]:
# Get basic cost response
cost_json = get_calculate_cost_basic_anthropic(
    model=response.model,
    input_tokens=response.usage.input_tokens,
    output_tokens=response.usage.output_tokens
)

# Show cost response
print("***** Basic Cost *****")
print(json.dumps(cost_json, indent=2, ensure_ascii=False))

***** Basic Cost *****
{
  "cost": {
    "currency_type": "USD",
    "model": "claude-haiku-4-5-20251001",
    "input": 1.7e-05,
    "output": 0.001895,
    "total": 0.001912
  }
}


### Modo 3: Cálculo avanzado CON función de costes específica

In [ ]:
# Get advance cost response
cost_json = get_calculate_cost_advance_anthropic(
    model=response.model,
    input_tokens=response.usage.input_tokens,
    output_tokens=response.usage.output_tokens
)

# Show cost response
print("***** Advance Cost *****")
print(json.dumps(cost_json, indent=2, ensure_ascii=False))

***** Advance Cost *****
{
  "cost": {
    "currency_type": "USD",
    "model": "claude-haiku-4-5-20251001",
    "input": 1.7e-05,
    "input_non_cached": 1.7e-05,
    "input_cache_creation": 0.0,
    "input_cached_read": 0.0,
    "output": 0.001895,
    "total": 0.001912
  }
}


### Modo 4: Cálculo avanzado CON función de costes genérica

In [ ]:
# Get cost response
cost_json = get_calculate_cost_anthropic(response)

# Show cost response
print("***** Generic Cost *****")
print(json.dumps(cost_json, indent=2, ensure_ascii=False))

***** Generic Cost *****
{
  "cost": {
    "currency_type": "USD",
    "model": "claude-haiku-4-5-20251001",
    "input": 1.7e-05,
    "input_non_cached": 1.7e-05,
    "input_cache_creation": 0.0,
    "input_cached_read": 0.0,
    "output": 0.001895,
    "total": 0.001912
  }
}


# System prompt

El "system prompt" facilita disponer de un contrato de calidad

Este prompt suele ir evolucionando con el tiempo

## Consideración 1: Uso de system prompt básico con Rol

La respuesta puede estar un poco más enfocada y ser un poco mejor a la invocación que solamente se utiliza el prompt del usuario (terminología más técnica, respuesta más acotada, etc)

Nota: Puede que NO haya mucha diferencia con el anterior caso





In [13]:
# Define system prompt: Role
SYSTEM_PROMPT = """
Acts as a senior software architect with experience in application modernization
"""

# Define User Prompt
USER_PROMPT = "What should I consider when modernizing an application?"

print("***** SYSTEM PROMPT - Consideración 1: Rol básico *****")

# Invoke LLM
response = anthropic_client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    system=SYSTEM_PROMPT,
    messages=[
        {"role": "user", "content": USER_PROMPT}
    ]
)

# Get text response
response_text = response.content[0].text

# Show
print("Respuesta del modelo:\n")
print(response_text)

# Get metadata response
metadata_json = get_response_metadata_basic_anthropic(response)

# Show metadata
print("\n***** Metadata *****")
print(json.dumps(metadata_json, indent=2, ensure_ascii=False))

# Get cost response
cost_json = get_calculate_cost_anthropic(response)

# Show cost response
print("\n***** Cost *****")
print(json.dumps(cost_json, indent=2, ensure_ascii=False))

***** SYSTEM PROMPT - Consideración 1: Rol básico *****
Respuesta del modelo:

# Application Modernization: Key Considerations

## 1. Assessment & Discovery First
- **Understand the current state** — document architecture, dependencies, tech debt
- **Identify pain points** — performance bottlenecks, scalability limits, maintenance costs
- **Map business criticality** — what breaks if this fails?
- **Audit integrations** — upstream/downstream dependencies often get underestimated

## 2. Define Clear Goals
Ask *why* you're modernizing:
- Cost reduction?
- Scalability?
- Developer velocity?
- Security/compliance?
- > Goals drive architecture decisions — don't modernize without them

## 3. Choosing a Strategy (the "6 Rs")
| Strategy | Description | When to Use |
|---|---|---|
| **Rehost** | Lift & shift | Quick wins, cost savings |
| **Replatform** | Minor optimizations | Managed services without rewrite |
| **Refactor** | Restructure code | Improve maintainability |
| **Re-architect** | C

## Consideración 2: Uso de system prompt con Rol + Audiencia + Formato de respuesta

Con esta opción ya si que se pueden poner parámetros que pueden afectar al coste

Ejemplo: Poner ejemplos de cada cosa (incrementa los tokens), limitar el formato de la respuesta (palabras, etc.)

El parámetro que más condiciona la respuesta es el formato de la respuesta ya que en la mayoría de los casos son restricciones

In [ ]:
# Define system prompt: Role + Audience + Format
SYSTEM_PROMPT = """
Acts as a senior software architect with experience in application modernization

The audience will be a team of backend developers with 5+ years of experience.

The response format will be:
- Write in prose
- Do not use tables, emojis, bullet points o numbered list
- Maximum of 300 words
"""

# Define User Prompt
USER_PROMPT = "What should I consider when modernizing an application?"

print("***** SYSTEM PROMPT - Consideración 2: Rol + Audiencia + Formato *****")

# Invoke LLM
response = anthropic_client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    system=SYSTEM_PROMPT,
    messages=[
        {"role": "user", "content": USER_PROMPT}
    ]
)

# Get text response
response_text = response.content[0].text

# Show
print("Respuesta del modelo:\n")
print(response_text)

# Get metadata response
metadata_json = get_response_metadata_basic_anthropic(response)

# Show metadata
print("\n***** Metadata *****")
print(json.dumps(metadata_json, indent=2, ensure_ascii=False))

# Get cost response
cost_json = get_calculate_cost_anthropic(response)

# Show cost response
print("\n***** Cost *****")
print(json.dumps(cost_json, indent=2, ensure_ascii=False))

***** SYSTEM PROMPT - Consideración 2: Rol + Audiencia + Formato *****
Respuesta del modelo:

# Application Modernization: Key Considerations

When modernizing an application, start by establishing clear business objectives alongside technical goals. Determine whether you're modernizing to reduce operational costs, improve time-to-market, enhance scalability, or increase maintainability. These drivers will shape your entire strategy and help justify the investment to stakeholders.

Assess your current system thoroughly. Understanding the architecture, dependencies, and data flows is crucial before making changes. Many teams underestimate the complexity hidden in legacy systems, particularly around implicit contracts between components and undocumented domain logic. This assessment should include evaluating technical debt, identifying performance bottlenecks, and documenting critical business processes.

Consider your team's capabilities and capacity. Modernization isn't purely technica

## Consideración 3: Uso de system prompt con Rol + Contexto + Audiencia + Formato de respuesta

Con esta opción podemos mejorar la calidad de la respuesta

Ejemplo: Poner ejemplos de cada cosa (incrementa los tokens), limitar el formato de la respuesta (palabras, etc.)

El "parámetro" que más condiciona la respuesta es el formato de la respuesta ya que en la mayoría de los casos son restricciones

In [ ]:
# Define system prompt: Role + Context + Tone + Audience + Format
SYSTEM_PROMPT = """
Act as a senior software architect with 15+ years of experience in application modernization.

Your goal is to help the user analyze, design, and propose practical technical solutions to modernize legacy or existing applications.

Remember to keep best practices in mind.

The audience will be a team of backend developers with 5+ years of experience.

The communication style will be didactic and professional, using clear explanations, a practical approach, and actionable recommendations.

Response rules:
- Write in prose
- Do not use tables, emojis, bullet points o numbered list
- Maximum of 300 words
"""

# Define User Prompt
USER_PROMPT = "What should I consider when modernizing an application?"

print("***** SYSTEM PROMPT - Consideración 3: Rol + Contexto + Tono + Audiencia + Formato *****")

# Invoke LLM
response = anthropic_client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    system=SYSTEM_PROMPT,
    messages=[
        {"role": "user", "content": USER_PROMPT}
    ]
)

# Get text response
response_text = response.content[0].text

# Show
print("Respuesta del modelo:\n")
print(response_text)

# Get metadata response
metadata_json = get_response_metadata_basic_anthropic(response)

# Show metadata
print("\n***** Metadata *****")
print(json.dumps(metadata_json, indent=2, ensure_ascii=False))

# Get cost response
cost_json = get_calculate_cost_anthropic(response)

# Show cost response
print("\n***** Cost *****")
print(json.dumps(cost_json, indent=2, ensure_ascii=False))

***** SYSTEM PROMPT - Consideración 3: Rol + Contexto + Tono + Audiencia + Formato *****
Respuesta del modelo:

# Key Considerations for Application Modernization

Modernizing an application requires a holistic approach that balances technical excellence with business reality. Start by understanding your current state deeply. Conduct a thorough assessment of your codebase, infrastructure, and dependencies. Identify technical debt hotspots, performance bottlenecks, and security vulnerabilities. This baseline becomes your reference point for measuring progress and justifying investments.

Next, align modernization with business objectives rather than pursuing technology for its own sake. Modernization should solve real problems: reducing operational costs, improving time-to-market, enabling new features, or enhancing reliability. This alignment ensures stakeholder support and helps prioritize efforts when resources are constrained.

Consider the strangler pattern as your primary strategy

## Consideración 4: Uso de system prompt con: Rol + Contexto + Audiencia + Formato de respuesta + Restricciones de comportamiento + Gestión de Casos Límite

Con esta opción podemos mejorar la calidad de la respuesta

Ejemplo: Poner ejemplos de cada cosa (incrementa los tokens), limitar el formato de la respuesta (palabras, etc.)

El "parámetro" que más condiciona la respuesta es el formato de la respuesta ya que en la mayoría de los casos son restricciones

In [ ]:
# Define system prompt: Role + Context + Tone + Audience + Format + Behavioral constraints + Edge case handling
SYSTEM_PROMPT = """
Act as a senior software architect with 15+ years of experience in application modernization.

Your goal is to help the user analyze, design, and propose practical technical solutions to modernize legacy or existing applications.

Remember to keep best practices in mind.

The audience will be a team of backend developers with 5+ years of experience.

The communication style will be didactic and professional, using clear explanations, a practical approach, and actionable recommendations.

Response rules:
- Write in prose
- Do not use tables, emojis, bullet points o numbered list
- Maximum of 300 words
- Prioritize actionable insights over generic theory
- Always mention at least one non-obvious risk that teams commonly overlook
- Do not recommend fashionable patterns such as microservices, event-driven architecture, Kubernetes or cloud migration unless they are justified by the application’s constraints, team maturity, operational model and business goals.
- If the user provides insufficient context about the legacy system, explicitly state the assumptions being made and propose a safe first step, such as assessment, observability, dependency mapping or incremental refactoring, instead of giving a definitive modernization plan.
"""

# Define User Prompt
USER_PROMPT = "What should I consider when modernizing an application?"

print("***** SYSTEM PROMPT - Consideración 4: Rol + Contexto + Tono + Audiencia + Formato + Restricciones de comportamiento + Gestión de Casos Límite *****")

# Invoke LLM
response = anthropic_client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    system=SYSTEM_PROMPT,
    messages=[
        {"role": "user", "content": USER_PROMPT}
    ]
)

# Get text response
response_text = response.content[0].text

# Show
print("Respuesta del modelo:\n")
print(response_text)

# Get metadata response
metadata_json = get_response_metadata_basic_anthropic(response)

# Show metadata
print("\n***** Metadata *****")
print(json.dumps(metadata_json, indent=2, ensure_ascii=False))

# Get cost response
cost_json = get_calculate_cost_anthropic(response)

# Show cost response
print("\n***** Cost *****")
print(json.dumps(cost_json, indent=2, ensure_ascii=False))

***** SYSTEM PROMPT - Consideración 4: Rol + Contexto + Tono + Audiencia + Formato + Restricciones de comportamiento + Gestión de Casos Límite *****
Respuesta del modelo:

Modernization without clear context is dangerous. Before architecting solutions, establish what "modernization" actually means for your situation, because the path differs radically depending on whether you're fixing performance, reducing operational burden, enabling feature velocity, or preparing for acquisition.

Start with a brutally honest assessment of three things: the business constraint driving this effort, the current system's actual bottleneck (which is rarely what people assume), and your team's operational readiness. I've seen organizations spend eighteen months on cloud migration only to realize their bottleneck was database queries, not infrastructure.

Map dependencies honestly. Build a dependency graph showing external integrations, data flows, and coupling points. This reveals where modernization cre

## Consideración 5: Uso de system prompt con contexto avanzado estructurado


In [ ]:
# Define system prompt with structure (Production-grade prompt)
PRODUCTION_SYSTEM_PROMPT = """
Act as a senior software architect with 15+ years of experience in application modernization.

# Objective
Your goal is to help the user analyze, design, and propose practical technical solutions to modernize legacy or existing applications, always considering software engineering best practices.

# Tone
- Didactic and professional
- Clear and structured explanations
- Practical and action-oriented
- Technically rigorous, but easy for experienced backend developers to follow

# Audience
- Backend developers with 5+ years of experience
- Technically proficient professionals familiar with software architecture, APIs, databases, deployment, and operational concerns

# Output Format
- Write in prose
- Do not use tables, emojis, bullet points o numbered list
- Maximum of 300 words
- Concise and actionable content
- Avoid generic statements; provide concrete technical insights

# Behavior Guidelines
- Prioritize clarity, precision, and real-world applicability
- Recommend solutions aligned with maintainability, scalability, security, observability, and operational simplicity
- Explain trade-offs when relevant
- Focus on practical modernization paths such as refactoring, modularization, API design, cloud readiness, CI/CD, testing, observability, and incremental migration
- Avoid over-explaining basic concepts unless necessary
- If key context is missing, make reasonable assumptions or ask concise, targeted questions
- Never repeat informaticion the user already provided

Do not:
- Use filler phrases like 'Great question!' or 'There are many things to consider'
- Repeat information the user already provided
"""

# Define User Prompt
USER_PROMPT = "What should I consider when modernizing an application?"

print("***** SYSTEM PROMPT - Consideración 5: Production-grade prompt *****")

# Invoke LLM
response = anthropic_client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    system=SYSTEM_PROMPT,
    messages=[
        {"role": "user", "content": USER_PROMPT}
    ]
)

# Get text response
response_text = response.content[0].text

# Show
print("Respuesta del modelo:\n")
print(response_text)

# Get metadata response
metadata_json = get_response_metadata_basic_anthropic(response)

# Show metadata
print("\n***** Metadata *****")
print(json.dumps(metadata_json, indent=2, ensure_ascii=False))

# Get cost response
cost_json = get_calculate_cost_anthropic(response)

# Show cost response
print("\n***** Cost *****")
print(json.dumps(cost_json, indent=2, ensure_ascii=False))

***** SYSTEM PROMPT - Consideración 5: Production-grade prompt *****
Respuesta del modelo:

Modernization is a strategic decision, not merely a technical one. Start by understanding what "modernization" actually means for your business. Are you accelerating feature delivery, improving operational reliability, reducing infrastructure costs, or making the codebase maintainable? Each driver leads to different technical paths, and conflating them creates waste.

Before touching the code, establish baseline observability. You need production metrics—response times, error rates, resource consumption, deployment frequency—to measure whether your modernization efforts actually deliver value. Teams often modernize blindly, only to discover the bottleneck was never the architecture.

Second, map dependencies and identify boundaries. Understand which parts of the system are tightly coupled, where data flows, and which components could realistically be decoupled or replaced incrementally. This det

# Parámetros del modelo

Existen una serie de características que permiten controlar el comportamiento' del modelo

Cada proveedor tiene sus propios parámetros aunque en muchos casos son compartidos


**IMPORTANTE**
* Los modelos de razonamiento NO suelen aceptar la mayoría de estos parámetros
* En muchos casos estos parámetros esta fijados internamento

## Consideración 1: Parámetro de temperatura

Determina la cantidad de aleatoriedad a la hora de elegir el siguiente token

Suele usarse para añadir determinismo o bien creatividad

En OpenAI se suele trabajar en un rango de: 0.0 a 2.0
En Anthropic se suele trabajar en un rango de: 0.0 a 1.0

Detalle:

* **0.0**: el modelo siempre elige el token más probable. Respuestas casi deterministas
* **0.3 - 0.5**: ligera variación. Bueno para análisis técnico, clasificación, extracción de datos
* **0.7 - 1.0** respuestas más variadas y creativas. Bueno para brainstorming, redacción creativa
* **> 1.0** — alta aleatoriedad. Puede producir resultados incoherentes

**TRUCOS**
* Se aconseja empezar con temperaturas de 0.2-0.3 en producción
* Ir subiendo poco a poco cuando se necesita "creatividad"


### Aplicación del parámetro "temperatura"

In [7]:
# Define system prompt: Role
SYSTEM_PROMPT = """
Acts as a senior software architect with experience in application modernization
"""

# Define User Prompt
USER_PROMPT = "What should I consider when modernizing an application?"

# Configure temperature (test with: 0.0, 0.5 y 1.0)
temperature = 0.3

print("***** PARAM - Consideración 1: Ejemplo de uso del parámetro temperatura *****")
print(f"Temperatura: {temperature}")

# Invoke LLM
response = anthropic_client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    temperature=temperature,
    system=SYSTEM_PROMPT,
    messages=[
        {"role": "user", "content": USER_PROMPT}
    ]
)

# Get text response
response_text = response.content[0].text

# Show
print("Respuesta del modelo:\n")
print(response_text)

# Get metadata response
metadata_json = get_response_metadata_basic_anthropic(response)

# Show metadata
print("\n***** Metadata *****")
print(json.dumps(metadata_json, indent=2, ensure_ascii=False))

# Get cost response
cost_json = get_calculate_cost_anthropic(response)

# Show cost response
print("\n***** Cost *****")
print(json.dumps(cost_json, indent=2, ensure_ascii=False))

***** PARAM - Consideración 1: Ejemplo de uso del parámetro temperatura *****
Temperatura: 0.3
Respuesta del modelo:

# Key Considerations for Application Modernization

## Strategic Assessment
- **Business Goals**: Align modernization with revenue, cost reduction, or competitive advantage objectives
- **Current State Analysis**: Document technical debt, performance bottlenecks, and security vulnerabilities
- **ROI Calculation**: Estimate costs vs. benefits over 3-5 years
- **Risk Tolerance**: Understand organizational appetite for disruption during transition

## Technical Architecture

**Migration Approach**
- **Lift & Shift**: Minimal changes, faster but limited benefits
- **Refactor**: Restructure code without changing functionality
- **Rearchitect**: Redesign for cloud-native, microservices, etc.
- **Rebuild**: Complete rewrite (highest risk/reward)

**Technology Choices**
- Cloud platform (AWS, Azure, GCP) vs. on-premises
- Containerization (Docker/Kubernetes) for portability
- M

### Comparativa de costes según diferentes combinaciones temperatura

In [8]:
# Define system prompt: Role
SYSTEM_PROMPT = """
Acts as a senior software architect with experience in application modernization
"""

# Define User Prompt
USER_PROMPT = "What should I consider when modernizing an application?"

# Configure temperature (test with: 0.0, 0.5 y 1.0)
temperatures = [0.0, 0.7, 1.0]

# Configura num excecutions
num_executions = 3

print("***** PARAM - Consideración 1: Comparativa de costes según la temperatura *****\n")

# Create table header
print(f"{'Temperature':>20s} {'Execution':>12s} {'Input tok':>12s} {'Output tok':>12s} {'Cost':>12s}")
print("-" * 80)

for temperature in temperatures:

    for i in range(num_executions):

      # Invoke LLM
      response = anthropic_client.messages.create(
          model="claude-haiku-4-5-20251001",
          max_tokens=1024,
          temperature=temperature,
          system=SYSTEM_PROMPT,
          messages=[
              {"role": "user", "content": USER_PROMPT}
          ]
      )

      # Get cost response
      cost_json = get_calculate_cost_anthropic(response)

      cost_str = f"${cost_json['cost']['total']:.6f}" if cost_json else "N/A"

      # Create table row
      print(f"{str(temperature):>20s} {str((i+1)):>12s} {response.usage.input_tokens:>12d} {response.usage.output_tokens:>12d} {cost_str:>12s}")


***** PARAM - Consideración 1: Comparativa de costes según la temperatura *****

         Temperature    Execution    Input tok   Output tok         Cost
--------------------------------------------------------------------------------
                 0.0            1           30          411    $0.002085
                 0.0            2           30          459    $0.002325
                 0.0            3           30          463    $0.002345
                 0.7            1           30          449    $0.002275
                 0.7            2           30          397    $0.002015
                 0.7            3           30          398    $0.002020
                 1.0            1           30          484    $0.002450
                 1.0            2           30          396    $0.002010
                 1.0            3           30          489    $0.002475


## Consideración 2: Consistencia de respuestas

La variabilidad de las respuestas puede ser muy alta debido principalmente a que los modelos son **estocásticos** (contiene un poco de azar o incertidumbre).

Por otro lado, hay que en cuenta la combinación: TEMPERATURA + INSTRUCCIONES_PRECISAS (USER_PROMPT)

Situaciones

* Temperatura elevada + Instrucción imprecisa = respuestas muy inconsistentes
* Temperatura baja + Instrucción precisa = respuestas consistentes y predecibles

NOTA: Este apartado NO tiene ejecicios/ejemplos asociados

## Consideración 3: Parámetro de máximo número de tokens

Determina la cantidad máxima de tokens que el modelo puede generar.

No se considera la longitud objetivo, se trata del límite superior.

Cuando se alcance este límite la respuesta se corta precipitadamente.

Se suele utilizar para controlar costes de los tokens de salida
Evita que el modelo genera respuestas excesivamente largas

**TRUCOS**
* Es un límite de seguridad -> NO un control de longitud
* Se combian con restricciones de formato

**IMPORTANTE**
* En OpenAI y Gemini es opcional
* En Anthropic es obligatorio (se usa max_tokens)
* Se conseja controlar el estado de salida de la petición ("status")

## Consideración 4: Parámetro de razonamiento

**IMPORTANTE**
* Se utilizará el modelo "gpt-5.4" que es un modelo con razonamiento

In [10]:
# Define system prompt: Role
SYSTEM_PROMPT = """
Acts as a senior software architect with experience in application modernization
"""

# Define User Prompt
USER_PROMPT = "What should I consider when modernizing an application?"

# Configure reasoning (test with: "low", "medium", "high" y "xhigh")
reasoning_effort = "medium"

# Map reasoning effort to Anthropic thinking budget
# budget_tokens must be lower than max_tokens
REASONING_BUDGETS = {
    "low": 1024,
    "medium": 2048,
    "high": 4096,
    "xhigh": 8192
}

reasoning_budget_tokens = REASONING_BUDGETS.get(reasoning_effort)

if reasoning_budget_tokens is None:
    raise ValueError(
        "Invalid reasoning_effort. Use one of: low, medium, high, xhigh"
    )

# max_tokens includes thinking tokens + final answer tokens
MAX_TOKENS = reasoning_budget_tokens + 2048

thinking_config = {
    "type": "enabled",
    "budget_tokens": reasoning_budget_tokens,
    "display": "summarized"
}

print("***** PARAM - Consideración 4: Ejemplo de uso del parámetro reasoning *****")
print(f"Reasoning effort: {reasoning_effort}")
print(f"Reasoning budget tokens: {reasoning_budget_tokens:,}")
print(f"Max tokens: {MAX_TOKENS:,}")

# Invoke LLM
response = anthropic_client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=MAX_TOKENS,
    # Do not pass temperature when thinking is enabled
    thinking=thinking_config,
    system=SYSTEM_PROMPT,
    messages=[
        {"role": "user", "content": USER_PROMPT}
    ]
)

# Extract response blocks
thinking_text_parts = []
response_text_parts = []

for block in response.content:
    if block.type == "thinking":
        thinking_text_parts.append(block.thinking)
    elif block.type == "text":
        response_text_parts.append(block.text)

thinking_text = "\n".join(thinking_text_parts)
response_text = "\n".join(response_text_parts)

# Show thinking summary
if thinking_text:
    print("\n***** Thinking summary *****\n")
    print(thinking_text)

# Show
print("\nRespuesta del modelo:\n")
print(response_text)

# Get metadata response
metadata_json = get_response_metadata_anthropic(response)

# Show metadata
print("\n***** Metadata *****")
print(json.dumps(metadata_json, indent=2, ensure_ascii=False))

# Get cost response
cost_json = get_calculate_cost_anthropic(response)

# Show cost response
print("\n***** Cost *****")
print(json.dumps(cost_json, indent=2, ensure_ascii=False))


***** PARAM - Consideración 4: Ejemplo de uso del parámetro reasoning *****
Reasoning effort: medium
Reasoning budget tokens: 2,048
Max tokens: 4,096

***** Thinking summary *****

The user is asking about application modernization considerations. As a senior software architect, I should provide comprehensive guidance covering the key aspects of application modernization. This is an important topic that affects business value, technical debt, and organizational capabilities.

Let me think through the main categories:

1. Business objectives
2. Technical assessment
3. Architecture decisions
4. Migration approach
5. Organizational/people factors
6. Risk management
7. Cost and timeline
8. Tools and technology choices

I'll organize this in a practical way that a decision-maker and technical team can use.

Respuesta del modelo:

# Key Considerations for Application Modernization

## **Business & Strategic**
- **Clear objectives**: Improved performance? Cost reduction? New capabilities? Fas

# Uso de diferentes modelos


## Consideración 1: Modelos diferentes dentro de un mismo proveedor

In [15]:
import json

# Define system prompt: Role
SYSTEM_PROMPT = """
Acts as a senior software architect with experience in application modernization
"""

# Define User Prompt
USER_PROMPT = "What should I consider when modernizing an application?"

# Configure Anthropic Models
anthropic_models = [
    "claude-haiku-4-5-20251001",
    "claude-sonnet-4-6",  # Uncomment if you have access
]

# Configure generation parameters
temperature = 0.3
max_tokens = 1024


def get_anthropic_response_text(response):
    """
    Extract only text blocks from an Anthropic response.

    This avoids errors when response.content contains non-text blocks,
    for example ThinkingBlock when reasoning/thinking is enabled.
    """
    text_parts = []

    for block in response.content:
        if getattr(block, "type", None) == "text":
            text_parts.append(block.text)

    return "\n".join(text_parts)


def get_anthropic_status_label(response):
    """
    Anthropic does not expose response.status like OpenAI.
    We use stop_reason to infer whether the answer completed or was truncated.
    """
    stop_reason = getattr(response, "stop_reason", None)

    if stop_reason == "max_tokens":
        return " [TRUNCATED]"

    if stop_reason in ("end_turn", "stop_sequence"):
        return " [COMPLETE]"

    return f" [STOP_REASON={stop_reason}]"


print("***** DIFERENTES MODELOS - Consideración 1: Modelos diferentes de un mismo proveedor *****")

for model_name in anthropic_models:

    print(f"\n ************************ ")
    print(f" [*] model = {model_name} ")
    print(f" ************************ ")

    try:
        # Invoke LLM
        response = anthropic_client.messages.create(
            model=model_name,
            max_tokens=max_tokens,
            temperature=temperature,
            system=SYSTEM_PROMPT,
            messages=[
                {"role": "user", "content": USER_PROMPT}
            ]
        )

        print(f"\n{'*' * 60}")

        # Usage details
        input_tokens = response.usage.input_tokens or 0
        output_tokens = response.usage.output_tokens or 0

        cache_creation_input_tokens = getattr(
            response.usage,
            "cache_creation_input_tokens",
            0
        ) or 0

        cache_read_input_tokens = getattr(
            response.usage,
            "cache_read_input_tokens",
            0
        ) or 0

        total_tokens = (
            input_tokens +
            output_tokens +
            cache_creation_input_tokens +
            cache_read_input_tokens
        )

        # Metadata details
        print(f"Modelo: {response.model}")
        print(
            "Tokens: "
            f"{input_tokens} input + "
            f"{output_tokens} output + "
            f"{cache_creation_input_tokens} cache_creation + "
            f"{cache_read_input_tokens} cache_read = "
            f"{total_tokens} tokens"
        )

        # Status details
        info_status = get_anthropic_status_label(response)
        print(f"Estado Respuesta: {info_status}")

        # Get cost response
        cost_json = get_calculate_cost_anthropic(response)

        # Cost details
        if cost_json:
            print(f"Coste: ${cost_json['cost']['total']:.6f}")
        else:
            print(f"Coste: No pricing found for model {response.model}")

        # Get metadata response
        metadata_json = get_response_metadata_anthropic(response)

        # Show metadata
        print("\n***** Metadata *****")
        print(json.dumps(metadata_json, indent=2, ensure_ascii=False))

        # Show cost response
        print("\n***** Cost *****")
        print(json.dumps(cost_json, indent=2, ensure_ascii=False))

        # Response details
        response_text = get_anthropic_response_text(response)

        print(f"\nRespuesta del modelo:\n{response_text}")

    except Exception as e:
        print(f"\n{'*' * 60}")
        print(f"Model {model_name}: Error — {e}")

***** DIFERENTES MODELOS - Consideración 1: Modelos diferentes de un mismo proveedor *****

 ************************ 
 [*] model = claude-haiku-4-5-20251001 
 ************************ 

************************************************************
Modelo: claude-haiku-4-5-20251001
Tokens: 30 input + 395 output + 0 cache_creation + 0 cache_read = 425 tokens
Estado Respuesta:  [COMPLETE]
Coste: $0.002005

***** Metadata *****
{
  "metadata": {
    "provider": "anthropic",
    "id": "msg_01Xs2LTt41MmThERFuiXXvkS",
    "model": "claude-haiku-4-5-20251001",
    "type": "message",
    "role": "assistant",
    "status": null,
    "created_at": null,
    "stop_reason": "end_turn",
    "stop_sequence": null,
    "usage": {
      "input_tokens": 30,
      "input_non_cached_tokens": 30,
      "input_cache_creation_tokens": 0,
      "input_cached_tokens": 0,
      "output_tokens": 395,
      "output_visible_tokens": 395,
      "output_reasoning_tokens": 0,
      "total_tokens": 425,
      "service

## Consideración 2: Modelos "equivalentes" en diferentes proveedores con las mismas condiciones de configuración

NOTA: Este apartado NO tiene ejecicios/ejemplos asociados

# Uso de herramientas

Resuelven ciertas situaciones o limitaciones que tienen los modelos

* Conocer cosas en fechas posteriores al final de su entrenamiento
* Acceso a información privada
* Ejecución de código

Son funcionalidades que el proveedor ejecuta en sus propios servidores: el modelo decide cuándo usarlas, las invoca, procesa los resultados y genera la respuesta final.

Hay de muchos tipos: búsqueda en internet, búsqueda en tus ficheros, ejecutar código, generación de imagenes, conectar son MCPs, etc.

**IMPORTANTE**
* NO todos los modelos los soportan
* Algunos de los tipos puede tener un coste adicional sobre el coste de los tokens normales (Revisar la actualización)


## Consideración 1: Uso de la herramienta "web_search"

Facilita al modelo a acceder a internet para buscar informacion que no tiene identificada en su entrenamiento

La característica tools=[{"type": "web_search_20250305", "name": "web_search", "max_uses": 3   # Cap on number of searches}] pone la herramienta a disposición del modelo, pero el modelo puede decidir si la usa o no según la consulta. La documentación indica explícitamente que, en Responses API, el modelo puede elegir buscar o no buscar en función del prompt

In [16]:
# Define system prompt: Role
SYSTEM_PROMPT = """
Acts as a senior software architect with experience in application modernization
"""

# Define User Prompt
WEB_USER_PROMPT = "What should I consider when modernizing an application in 2027?"

# Invoke LLM
response = anthropic_client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    temperature=0.3,
    system=SYSTEM_PROMPT,
    messages=[
        {"role": "user", "content": USER_PROMPT}
    ],
    tools=[{
        "type": "web_search_20250305",
        "name": "web_search",
        "max_uses": 3   # Cap on number of searches
    }]
)

# Get text response
response_text = response.content[0].text

# Show
print("Respuesta del modelo:\n")
print(response_text)

# Get metadata response
metadata_json = get_response_metadata_anthropic(response)

# Show metadata
print("\n***** Metadata *****")
print(json.dumps(metadata_json, indent=2, ensure_ascii=False))

# Get cost response
cost_json = get_calculate_cost_anthropic(response)

# Show cost response
print("\n***** Cost *****")
print(json.dumps(cost_json, indent=2, ensure_ascii=False))

Respuesta del modelo:

# Key Considerations for Application Modernization

As a senior software architect, here are the critical factors you should evaluate when modernizing an application:

## 1. **Business Objectives & Strategy**
- Define clear goals: cost reduction, improved performance, faster time-to-market, or enhanced user experience
- Align modernization efforts with overall business strategy
- Establish measurable success metrics and ROI expectations

## 2. **Current State Assessment**
- Conduct a thorough audit of existing systems, architecture, and dependencies
- Identify technical debt, performance bottlenecks, and security vulnerabilities
- Document business-critical processes and data flows
- Evaluate code quality, maintainability, and test coverage

## 3. **Modernization Approach**
Choose an appropriate strategy:
- **Rehost (Lift & Shift)**: Move to cloud with minimal changes
- **Replatform**: Make optimizations during migration
- **Refactor/Re-architect**: Restructure f

In [18]:
# The response may contain multiple content blocks (text + tool_use + text)
for block in response.content:
    if block.type == "text":
        print(block.text)

# =========================
# SAFE TOOL USAGE CHECK
# =========================

server_tool_use = getattr(
    response.usage,
    "server_tool_use",
    None
)

if server_tool_use is not None:

    web_searches = getattr(
        server_tool_use,
        "web_search_requests",
        0
    )

    print(f"\n[Web searches performed: {web_searches}]")

else:

    print("\n[Web searches performed: 0]")

# =========================
# TOKEN USAGE
# =========================

print(
    f"[Tokens: "
    f"{response.usage.input_tokens} in / "
    f"{response.usage.output_tokens} out]"
)

# Key Considerations for Application Modernization

As a senior software architect, here are the critical factors you should evaluate when modernizing an application:

## 1. **Business Objectives & Strategy**
- Define clear goals: cost reduction, improved performance, faster time-to-market, or enhanced user experience
- Align modernization efforts with overall business strategy
- Establish measurable success metrics and ROI expectations

## 2. **Current State Assessment**
- Conduct a thorough audit of existing systems, architecture, and dependencies
- Identify technical debt, performance bottlenecks, and security vulnerabilities
- Document business-critical processes and data flows
- Evaluate code quality, maintainability, and test coverage

## 3. **Modernization Approach**
Choose an appropriate strategy:
- **Rehost (Lift & Shift)**: Move to cloud with minimal changes
- **Replatform**: Make optimizations during migration
- **Refactor/Re-architect**: Restructure for cloud-native capabil

In [19]:
# =========================
# EXTRACT URLS
# =========================

urls_found = []

for block in response.content:

    # Texto normal
    if block.type == "text":

        print("\nTEXT BLOCK:")
        print(block.text)

        # Algunas respuestas incluyen citas
        citations = getattr(block, "citations", None)

        if citations:

            for citation in citations:

                url = getattr(citation, "url", None)

                if url:
                    urls_found.append(url)

    # Resultados de herramienta
    elif "tool" in block.type:

        print("\nTOOL BLOCK:")
        print(block)

# =========================
# SHOW URLS
# =========================

if urls_found:

    print("\n===== URLS FOUND =====")

    for url in set(urls_found):
        print(url)

else:

    print("\nNo URLs found.")


TEXT BLOCK:
# Key Considerations for Application Modernization

As a senior software architect, here are the critical factors you should evaluate when modernizing an application:

## 1. **Business Objectives & Strategy**
- Define clear goals: cost reduction, improved performance, faster time-to-market, or enhanced user experience
- Align modernization efforts with overall business strategy
- Establish measurable success metrics and ROI expectations

## 2. **Current State Assessment**
- Conduct a thorough audit of existing systems, architecture, and dependencies
- Identify technical debt, performance bottlenecks, and security vulnerabilities
- Document business-critical processes and data flows
- Evaluate code quality, maintainability, and test coverage

## 3. **Modernization Approach**
Choose an appropriate strategy:
- **Rehost (Lift & Shift)**: Move to cloud with minimal changes
- **Replatform**: Make optimizations during migration
- **Refactor/Re-architect**: Restructure for cloud-n

# Escalado de Costes

### Aplicación de una conversación multi-turno

In [21]:
# Define system prompt: Role
SYSTEM_PROMPT = """
Acts as a senior software architect with experience in application modernization.

You are helping a team to devise a strategy for modernising an application.
Ask clarifying questions where necessary.
Keep each answer to a maximum of three sentences.
"""

# Configure conversation history
conversation_history = []

# Configure prepared questions
questions = [
    "What are the main business goals driving this application modernization initiative?",
    "Can you help us assess the current state of the application, including architecture, technologies, dependencies, and pain points?",
    "Which parts of the application should we prioritize based on business criticality, technical risk, and modernization value?",
    "What modernization strategy would you recommend for this application: rehost, replatform, refactor, rearchitect, rebuild, or replace?",
    "What should the initial modernization roadmap look like, including phases, risks, quick wins, required team skills, and success metrics?",
]

print("***** ESCALADO - Consideración 1: Coste en conversaciones multi-turno *****\n")

# Print table header
print(
    f"{'Turn':>6s} "
    f"{'Input tok':>12s} "
    f"{'Output tok':>12s} "
    f"{'Cumulative cost':>18s}   "
    f"{'Question':>6s}"
)

print("-" * 100)

cumulative_cost = 0

for i, question in enumerate(questions):

    ## Add Question
    conversation_history.append({
        "role": "user",
        "content": question
    })

    try:
        # Invoke LLM
        response = anthropic_client.messages.create(
            model="claude-haiku-4-5-20251001",
            system=SYSTEM_PROMPT,
            messages=conversation_history,
            max_tokens=1024,
            temperature=0.3
        )

        # Extract text response
        response_text_parts = []

        for block in response.content:

            if getattr(block, "type", None) == "text":
                response_text_parts.append(block.text)

        response_text = "\n".join(response_text_parts)

        ## Add Response
        conversation_history.append({
            "role": "assistant",
            "content": response_text
        })

        # =========================
        # TOKEN USAGE
        # =========================

        input_tokens = response.usage.input_tokens or 0
        output_tokens = response.usage.output_tokens or 0

        # =========================
        # COST
        # =========================

        cost_json = get_calculate_cost_anthropic(response)

        cost_str = "N/A"

        if cost_json:

            cumulative_cost += cost_json["cost"]["total"]

            cost_str = f"${cumulative_cost:.6f}"

        # Print row
        print(
            f"{i+1:>6d} "
            f"{input_tokens:>12d} "
            f"{output_tokens:>12d} "
            f"{cost_str:>18s}   "
            f"{question[:40]:>40s}"
        )

    except Exception as e:

        print(
            f"{i+1:>6d} "
            f"{'ERROR':>12s} "
            f"{'ERROR':>12s} "
            f"{'ERROR':>18s}   "
            f"{str(e)}"
        )

***** ESCALADO - Consideración 1: Coste en conversaciones multi-turno *****

  Turn    Input tok   Output tok    Cumulative cost   Question
----------------------------------------------------------------------------------------------------
     1           69           78          $0.000459   What are the main business goals driving
     2          173          154          $0.001402   Can you help us assess the current state
     3          353          139          $0.002450   Which parts of the application should we
     4          526          167          $0.003811   What modernization strategy would you re
     5          723          217          $0.005619   What should the initial modernization ro


### Sobrecoste del systen prompt

Esto se explica mejor en la parte de OpenAI. Para resolver las dudas o mirar explicaciónes revisar ese apartado

## Consideración 2: Coste de escalado real

La cantidad de tokenes utilizados NO son muy grandes para la realidad actual

In [22]:
# Configure general parameters
USERS = 1_000
DAYS_PER_MONTH = 22

# Configure calls parameters
CALLS_PER_USER_PER_DAY = 8
CALLS_PER_DAY = USERS * CALLS_PER_USER_PER_DAY
CALLS_PER_MONTH = CALLS_PER_DAY * DAYS_PER_MONTH

# Configure the average number of tokens used per cal
AVG_INPUT_TOKENS = 30_000
AVG_OUTPUT_TOKENS = 1_500

# Calculate monthly token estimation
INPUT_TOKENS_PER_MONTH = CALLS_PER_MONTH * AVG_INPUT_TOKENS
OUTPUT_TOKENS_PER_MONTH = CALLS_PER_MONTH * AVG_OUTPUT_TOKENS
TOTAL_TOKENS_PER_MONTH = INPUT_TOKENS_PER_MONTH + OUTPUT_TOKENS_PER_MONTH

# Configure available models
# - Prices are expressed in USD per 1M tokens
models_to_compare = {
    "claude-haiku-4-5-20251001": {"input": 1.00,"output": 5.00},
    "claude-sonnet-4-6": {"input": 3.00,"output": 15.00},
    "claude-opus-4-7": {"input": 5.00,"output": 25.00},
}

print("***** ESCALADO - Consideración 2: Coste de escalado real *****\n")

print("Objetivo:")
print("Este ejemplo estima el coste mensual de uso de varios modelos LLM")
print("en un escenario donde múltiples usuarios realizan llamadas diarias a la API.\n")

print("***** Detalles del escenario *****")
print(f"- Número de usuarios: {USERS:,} usuarios")
print(f"- Días laborables estimados al mes: {DAYS_PER_MONTH} días")
print(f"- Llamadas por usuario y día: {CALLS_PER_USER_PER_DAY} llamadas/día")
print(f"- Total de llamadas por día: {CALLS_PER_DAY:,} llamadas/día")
print(f"- Total de llamadas por mes: {CALLS_PER_MONTH:,} llamadas/mes")

print("\n***** Tokens estimados por llamada *****")
print(f"- Tokens de entrada por llamada: {AVG_INPUT_TOKENS:,} input tokens")
print(f"- Tokens de salida por llamada: {AVG_OUTPUT_TOKENS:,} output tokens")

print("\n***** Tokens estimados por mes*****")
print(f"- Tokens de entrada al mes: {INPUT_TOKENS_PER_MONTH:,} input tokens")
print(f"- Tokens de salida al mes: {OUTPUT_TOKENS_PER_MONTH:,} output tokens")
print(f"- Tokens totales al mes: {TOTAL_TOKENS_PER_MONTH:,} tokens")

print("\n***** Fórmula de cálculo *****")
print("Coste por llamada =")
print("  (input_tokens / 1,000,000 * precio_input) +")
print("  (output_tokens / 1,000,000 * precio_output)")
print("\nCoste mensual = coste por llamada * total de llamadas al mes")

print("\n**** Calculo de costes de escalado *****")

# Create table header
print(
    f"\n{'Model':25s} "
    f"{'Cost/call':>15s} "
    f"{'Cost/day':>15s} "
    f"{'Cost/month':>15s} "
    f"{'Cost/user/month':>18s}"
)
print("-" * 95)

for model_name, prices in models_to_compare.items():

    input_cost_per_call = (AVG_INPUT_TOKENS / 1_000_000) * prices["input"]
    output_cost_per_call = (AVG_OUTPUT_TOKENS / 1_000_000) * prices["output"]
    cost_per_call = input_cost_per_call + output_cost_per_call

    daily_cost = cost_per_call * CALLS_PER_DAY
    monthly_cost = cost_per_call * CALLS_PER_MONTH
    monthly_cost_per_user = monthly_cost / USERS

    print(
        f"{model_name:25s} "
        f"{money(cost_per_call, 6):>15s} "
        f"{money(daily_cost, 2):>15s} "
        f"{money(monthly_cost, 2):>15s} "
        f"{money(monthly_cost_per_user, 2):>18s}"
    )

print("\n***** Lectura rápida *****")
print("- Cost/call indica cuánto cuesta una llamada media al modelo")
print("- Cost/day indica el coste diario para todos los usuarios")
print("- Cost/month indica el coste mensual total estimado")
print("- Cost/user/month indica el coste mensual medio por usuario")

***** ESCALADO - Consideración 2: Coste de escalado real *****

Objetivo:
Este ejemplo estima el coste mensual de uso de varios modelos LLM
en un escenario donde múltiples usuarios realizan llamadas diarias a la API.

***** Detalles del escenario *****
- Número de usuarios: 1,000 usuarios
- Días laborables estimados al mes: 22 días
- Llamadas por usuario y día: 8 llamadas/día
- Total de llamadas por día: 8,000 llamadas/día
- Total de llamadas por mes: 176,000 llamadas/mes

***** Tokens estimados por llamada *****
- Tokens de entrada por llamada: 30,000 input tokens
- Tokens de salida por llamada: 1,500 output tokens

***** Tokens estimados por mes*****
- Tokens de entrada al mes: 5,280,000,000 input tokens
- Tokens de salida al mes: 264,000,000 output tokens
- Tokens totales al mes: 5,544,000,000 tokens

***** Fórmula de cálculo *****
Coste por llamada =
  (input_tokens / 1,000,000 * precio_input) +
  (output_tokens / 1,000,000 * precio_output)

Coste mensual = coste por llamada * tota

## Consideración 3: Crecimiento del contexto en conversaciones multi-turno

La cantidad de tokenes utilizados NO son muy grandes para la realidad actual

In [23]:
# Configure scenario
USERS = 1_000
CONVERSATIONS_PER_USER_PER_MONTH = 20
TURNS_PER_CONVERSATION = 12

# Token assumptions
SYSTEM_PROMPT_TOKENS = 350
AVG_USER_MESSAGE_TOKENS = 80
AVG_ASSISTANT_RESPONSE_TOKENS = 200

# Model pricing in USD per 1M tokens
MODEL_NAME = "claude-haiku-4-5-20251001"
INPUT_PRICE_PER_1M = 1.00
OUTPUT_PRICE_PER_1M = 5.00

TOTAL_CONVERSATIONS_PER_MONTH = USERS * CONVERSATIONS_PER_USER_PER_MONTH

print("***** ESCALADO - Consideración 3: Escalado de coste en conversaciones multi-turno *****\n")

print("***** Detalles del escenario *****")
print(f"- Usuarios: {USERS:,}")
print(f"- Conversaciones por usuario al mes: {CONVERSATIONS_PER_USER_PER_MONTH}")
print(f"- Turnos por conversación: {TURNS_PER_CONVERSATION}")
print(f"- Conversaciones totales al mes: {TOTAL_CONVERSATIONS_PER_MONTH:,}")
print(f"- Modelo seleccionado: {MODEL_NAME}")

print("\n***** Supuestos de tokens *****")
print(f"- System prompt: {SYSTEM_PROMPT_TOKENS:,} tokens")
print(f"- Mensaje medio del usuario: {AVG_USER_MESSAGE_TOKENS:,} tokens")
print(f"- Respuesta media del asistente: {AVG_ASSISTANT_RESPONSE_TOKENS:,} tokens")

print("\n***** Simulación de una conversación *****")
print(
    f"\n{'Turn':>6s} "
    f"{'Input tokens':>15s} "
    f"{'Output tokens':>15s} "
    f"{'Cost/turn':>15s}"
)
print("-" * 60)

conversation_input_tokens = 0
conversation_output_tokens = 0
conversation_cost = 0

for turn in range(1, TURNS_PER_CONVERSATION + 1):

    # In each turn we resend:
    # - system prompt
    # - all previous user messages
    # - all previous assistant responses
    # - current user message
    input_tokens = (
        SYSTEM_PROMPT_TOKENS +
        turn * AVG_USER_MESSAGE_TOKENS +
        (turn - 1) * AVG_ASSISTANT_RESPONSE_TOKENS
    )

    output_tokens = AVG_ASSISTANT_RESPONSE_TOKENS

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_1M
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_1M
    turn_cost = input_cost + output_cost

    conversation_input_tokens += input_tokens
    conversation_output_tokens += output_tokens
    conversation_cost += turn_cost

    print(
        f"{turn:>6d} "
        f"{input_tokens:>15,d} "
        f"{output_tokens:>15,d} "
        f"{money(turn_cost, 8):>18s}"
    )

monthly_cost = conversation_cost * TOTAL_CONVERSATIONS_PER_MONTH

print("\n***** Resumen por conversación *****")
print(f"- Input tokens por conversación: {conversation_input_tokens:,}")
print(f"- Output tokens por conversación: {conversation_output_tokens:,}")
print(f"- Coste por conversación: ${conversation_cost:.6f}")

print("\n***** Estimación mensual *****")
print(f"- Conversaciones al mes: {TOTAL_CONVERSATIONS_PER_MONTH:,}")
print(f"- Coste mensual estimado: ${monthly_cost:,.2f}")

print("\n***** Lectura rápida *****")
print("- El coste por turno aumenta porque cada nueva llamada incluye más historial")
print("- Las conversaciones largas pueden ser mucho más caras que llamadas aisladas")
print("- Una estrategia de resumen de contexto puede reducir costes en escenarios multi-turno")

***** ESCALADO - Consideración 3: Escalado de coste en conversaciones multi-turno *****

***** Detalles del escenario *****
- Usuarios: 1,000
- Conversaciones por usuario al mes: 20
- Turnos por conversación: 12
- Conversaciones totales al mes: 20,000
- Modelo seleccionado: claude-haiku-4-5-20251001

***** Supuestos de tokens *****
- System prompt: 350 tokens
- Mensaje medio del usuario: 80 tokens
- Respuesta media del asistente: 200 tokens

***** Simulación de una conversación *****

  Turn    Input tokens   Output tokens       Cost/turn
------------------------------------------------------------
     1             430             200        $0.00143000
     2             710             200        $0.00171000
     3             990             200        $0.00199000
     4           1,270             200        $0.00227000
     5           1,550             200        $0.00255000
     6           1,830             200        $0.00283000
     7           2,110             200        